# QLoRA Qwen2.5-VL-3B → GGUF для Ollama. Kaggle, финальная версия (v4)

Всё, что было выучено на предыдущих итерациях, уже внутри:
- обучающий датасет **сбалансирован** (дефекты + good), промпты совпадают с бенчмарком дословно
- 12 эпох (~330 шагов), адаптер сохраняется **сразу** после обучения
- merge на **CPU** (fp16-база не влезает на T4 рядом с тренировочной моделью)
- текстовый GGUF конвертируется свежим llama.cpp из смерженной модели (unsloth-экспорт текста пропускает LoRA — проверено хэшами)
- mmproj: попытка unsloth-экспорта; если не выйдет — используется проверенный mmproj из прогона v2 (уже лежит у тебя в `models\`)

Результат: `defect_vlm_q8_0.gguf` (текст, обязателен) + mmproj (если экспортируется).

**Подготовка:** обнови датасет — новый `vlm_finetune_train.jsonl` (178 записей) из `artifacts\`, плюс `mvtec_images.zip` (скриптом `pack_dataset_zip.py`). Runtime: T4 x2, Internet On.

In [ ]:
# 1. Установка Unsloth (~4 минуты)
%pip install --upgrade --no-cache-dir -q unsloth unsloth_zoo
%pip install --upgrade --no-deps --no-cache-dir -q bitsandbytes accelerate peft trl triton

In [ ]:
# 2. Данные: с диска, если уже загружены, иначе через upload.
import glob
import os
import shutil

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

def find_file(pattern, target):
    if os.path.exists(target):
        return target
    matches = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime, reverse=True)
    if matches:
        shutil.copy2(matches[0], target)  # input is read-only - copy, not move
        return target
    return None

jsonl_path = find_file("/kaggle/input/**/vlm_finetune_train*.jsonl", f"{WORK}/vlm_finetune_train.jsonl")
zip_path = find_file("/kaggle/input/**/mvtec_images*.zip", f"{WORK}/mvtec_images.zip")
if jsonl_path is None or zip_path is None:
    from google.colab import files
    files.upload()
    jsonl_path = find_file("/kaggle/input/**/vlm_finetune_train*.jsonl", f"{WORK}/vlm_finetune_train.jsonl")
    zip_path = find_file("/kaggle/input/**/mvtec_images*.zip", f"{WORK}/mvtec_images.zip")
assert jsonl_path and zip_path, "Нет vlm_finetune_train.jsonl и/или mvtec_images.zip в Input"
print("OK:", jsonl_path, "|", zip_path)

In [ ]:
# 3. Распаковка (Kaggle мог распаковать сам - тогда пропустит)
import os

if os.path.isdir(f"{WORK}/mvtec_anomaly_detection"):
    print("Images in place:", os.listdir(f"{WORK}/mvtec_anomaly_detection"))
else:
    !unzip -q -o {WORK}/mvtec_images.zip -d {WORK}
    !ls {WORK}/mvtec_anomaly_detection

In [ ]:
# 4. Модель в 4-бит + LoRA
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-3B-Instruct-unsloth-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# 5. Датасет. Промпты ДОСЛОВНО те же, что шлёт бенчмарк (vlm.py).
import json
from pathlib import Path

from datasets import Dataset
from PIL import Image

SYSTEM = "You are an industrial quality-control inspector. You analyze device and product images and report defects. Answer strictly in JSON matching the requested schema. Field meanings: defect_type is a short defect class name in English, or 'good' when no defect is visible; location describes where the defect is (e.g. 'bottom left, near the cap') or 'none'; severity is one of 'none', 'minor', 'major', 'critical'; confidence is a number between 0 and 1."
USER = "Inspect this image. Report whether the object is normal or defective, the defect type, its location, severity and your confidence. Reply with JSON only."
CONTENT_ROOT = Path(WORK)

def convert(record):
    image_path = Path(str(CONTENT_ROOT / record["image"]).replace("\\", "/"))
    assert image_path.exists(), f"Missing image: {image_path}"
    return {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": [
                {"type": "image", "image": Image.open(image_path).convert("RGB")},
                {"type": "text", "text": USER},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": record["messages"][2]["content"]},
            ]},
        ]
    }

records = [json.loads(line) for line in open(jsonl_path, encoding="utf-8") if line.strip()]
n_def = sum(1 for r in records if json.loads(r["messages"][2]["content"])["is_defect"])
print(f"{len(records)} records: {n_def} defective, {len(records) - n_def} good")
assert 0 < n_def < len(records), "Датасет должен быть сбалансирован (new vlm_finetune_train.jsonl!)"
train_ds = Dataset.from_list([convert(r) for r in records]).shuffle(seed=42)
train_ds[0]["messages"][1]["content"][0]

In [ ]:
# 6. Обучение: 12 эпох (~330 шагов). Loss должен упасть ниже ~0.05.
# Если CUDA OOM: per_device_train_batch_size=1.
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=12,
        warmup_steps=5,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=f"{WORK}/qlora_defect",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)
FastVisionModel.for_training(model)
trainer.train()

In [ ]:
# 7. СОХРАНЕНИЕ адаптера сразу после обучения (веса живут только в VRAM)
import os, shutil

model.save_pretrained(f"{WORK}/defect_lora_adapter")
processor.save_pretrained(f"{WORK}/defect_lora_adapter")
print("Adapter saved:", os.listdir(f"{WORK}/defect_lora_adapter"))
# освободить место: чекпоинты тренера не нужны
shutil.rmtree(f"{WORK}/qlora_defect", ignore_errors=True)
shutil.rmtree(f"{WORK}/mvtec_images.zip", ignore_errors=True)

In [ ]:
# 8. MERGE на CPU: вшиваем адаптер в fp16-базу (transformers+PEFT, без unsloth).
# На GPU не влезает: 4-битная тренировочная модель держит ~10GB из 14.5GB T4.
import torch
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor

base = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="cpu",
)
ft = PeftModel.from_pretrained(base, f"{WORK}/defect_lora_adapter")
merged = ft.merge_and_unload()
merged.save_pretrained(f"{WORK}/merged_model", safe_serialization=True)
del base, ft, merged
AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct").save_pretrained(f"{WORK}/merged_model")

# ФИКС: unsloth-процессор сохраняет extra_special_tokens списком, свежий transformers
# ожидает dict - конвертер падает на чтении. Правим сразу.
cfg_path = f"{WORK}/merged_model/tokenizer_config.json"
cfg = json.load(open(cfg_path, encoding="utf-8"))
if isinstance(cfg.get("extra_special_tokens"), list):
    cfg["extra_special_tokens"] = {}
    json.dump(cfg, open(cfg_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("tokenizer_config.json: extra_special_tokens fixed")
print("Merged model ready")

In [ ]:
# 9. ТЕКСТОВЫЙ GGUF: свежий llama.cpp (не unsloth - тот пропускает текстовый LoRA).
import glob
import os
import shutil

LLAMA_SRC = f"{WORK}/llama.cpp"
if not os.path.exists(f"{LLAMA_SRC}/convert_hf_to_gguf.py"):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_SRC}
    !pip install -q -r {LLAMA_SRC}/requirements.txt

!python {LLAMA_SRC}/convert_hf_to_gguf.py {WORK}/merged_model --outfile {WORK}/defect_vlm_q8_0.gguf --outtype q8_0
text_size = os.path.getsize(f"{WORK}/defect_vlm_q8_0.gguf")
assert text_size > 1_000_000_000, f"text gguf too small: {text_size}"
print(f"TEXT OK: {text_size/1e9:.1f} GB")
# merged_model больше не нужен - освобождаем 7GB перед unsloth-экспортом
shutil.rmtree(f"{WORK}/merged_model", ignore_errors=True)

In [ ]:
# 10. MMPROJ: пробуем unsloth-экспорт (в v2 он давал корректный обученный mmproj).
# НЕ фатально: если не выйдет - берём проверенный mmproj из прогона v2 (уже в models\).
import glob
import os

mmproj_files = []
try:
    model.save_pretrained_gguf(f"{WORK}/unsloth_gguf", processor, quantization_method="q8_0")
    mmproj_files = [f for f in glob.glob(f"{WORK}/unsloth_gguf/**/*.gguf", recursive=True)
                    if "mmproj" in os.path.basename(f).lower() and os.path.getsize(f) > 100_000_000]
except Exception as exc:
    print("unsloth export failed (non-fatal):", exc)

if mmproj_files:
    print("MMPROJ OK - скачивай:", mmproj_files[0])
else:
    print()
    print(">>> MMPROJ НЕ ЭКСПОРТИРОВАЛСЯ. Скачай ТОЛЬКО defect_vlm_q8_0.gguf.")
    print(">>> Как mmproj используем проверенный из прогона v2:")
    print(">>> models\\qwen2.5-vl-3b-instruct.F16-mmproj.gguf (b30dc4b1...) - он уже у тебя.")

## Итог

Скачай из Output:
1. **`defect_vlm_q8_0.gguf`** — дообученный текст (обязательно)
2. **mmproj** — если ячейка 10 напечатала `MMPROJ OK - скачивай`: файл из `unsloth_gguf/`. Если напечатала fallback — ничего не качай для vision, используем `models\qwen2.5-vl-3b-instruct.F16-mmproj.gguf` (v2)

Дома:
```
# Modelfile:
# FROM ./defect_vlm_q8_0.gguf
# FROM ./mmproj.gguf
# PARAMETER temperature 0
ollama create defect-qwen-vl -f Modelfile
```
Финальную проверку и бенчмарк делает Kilo локально.